# **BYTE PAIR ENCODING (BPE) - TOKENIZER**


---



## 1. **Character-level Tokenization:**

65 unique characters, each mapped to an integer. `stoi`/`itos`, `encode()`/`decode()` — simple.

But character-level tokenization has a real ceiling:

- Every character is a separate prediction. The word "Shakespeare" is 11 separate tokens the model has to get right in sequence.
- The model spends its limited capacity (128-dim embeddings, 4 layers) learning to spell common words letter-by-letter, instead of learning *meaning* and *structure*.
- Your context window (`block_size = 64`) only covers ~64 characters — roughly 10-12 words. A subword tokenizer would let the same 64 "slots" cover 3-4x more actual text.

## 2. What **Subword Tokenization** actually is

The core idea: don't tokenize by character (too fine, too many tokens) and don't tokenize by whole word (too coarse, vocabulary explodes — every conjugation, typo, and rare word needs its own slot). Split the difference: tokenize by **frequently-occurring chunks of characters**.

Concretely, a real BPE tokenizer would encode:
- `"the"` → one token (extremely common, gets its own slot)
- `"tokenization"` → maybe `["token", "ization"]` (two tokens — common suffix "ization" is shared across thousands of words)
- `"Grzegorzewski"` (a name it's never seen) → falls back to something like `["G", "rz", "ego", "rz", "ew", "ski"]` — never *fails*, just degrades gracefully to smaller pieces.

## 3. **Byte Pair Encoding** — the algorithm behind it

BPE is how you *build* that vocabulary of chunks. The algorithm, from scratch:

1. **Start at the character level.** Every character is its own token. (Sound familiar? This is literally your current setup — BPE starts exactly where MiniGPT already is.)
2. **Count every adjacent pair of tokens** in your training text. E.g., in "the cat sat", pairs are `(t,h)`, `(h,e)`, `(e,space)`, `(space,c)`, `(c,a)`, `(a,t)`, ...
3. **Find the single most frequent pair** across the whole corpus. Say `(t, h)` appears 10,000 times — more than any other pair.
4. **Merge that pair into one new token.** Every `t` immediately followed by `h` becomes a single unit `th`. Vocabulary grows by 1 (now has `th` in addition to `t` and `h`).
5. **Repeat.** Recount pairs (now including pairs involving your new `th` token, e.g., `(th, e)`), find the new most frequent pair, merge it. Maybe `th`+`e` → `the` becomes its own token next.
6. **Stop after N merges** — N is a hyperparameter you choose upfront. It directly determines your final vocab size: `final_vocab_size = starting_chars + N merges`.

So if you start with 65 characters and do, say, 1,935 merges, you land on a 2,000-token vocabulary. Every merge you perform makes one more "chunk" a single token instead of multiple characters — that's literally where the compression comes from.

---

## **What we're building**
A from-scratch BPE trainer — it takes your corpus (Tiny Shakespeare) and runs the merge loop from last session (count pairs → merge most frequent → repeat) for N iterations, producing your own vocabulary and merge rules. Then an encoder/decoder that applies those learned merges to any new text.

We start from bytes, not characters — same trick GPT-2 uses. `text.encode("utf-8")` turns any string into raw bytes `0-255`. This means our starting vocabulary is always exactly 256 tokens.


---



### **Step 1** — represent text as a list of tokens, byte-level

In [ ]:
def get_stats(ids):
    """
    Count how often every adjacent pair occurs.
    ids: list of ints (current token sequence)
    Returns: dict {(id1, id2): count}
    """
    counts = {}
    for pair in zip(ids, ids[1:]):          # (ids[0],ids[1]), (ids[1],ids[2]), ...
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    """
    Replace every occurrence of `pair` in ids with a single new_id.
    ids: list of ints
    pair: (id1, id2) tuple to merge
    new_id: the new token id to substitute
    """
    new_ids = []
    i = 0
    while i < len(ids):
        # Check if this pair matches at position i
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2                           # skip both merged tokens
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

### **Step 2** — the training loop


In [ ]:
def train_bpe(text, vocab_size):
    """
    text: full training corpus (string)
    vocab_size: target vocab size (must be > 256)
    Returns: merges dict {(id1,id2): new_id}, in the order they were learned
    """
    num_merges = vocab_size - 256
    ids = list(text.encode("utf-8"))       # start: raw bytes, 0-255
    print("IDS: ")
    print(ids)

    merges = {}   # (id1, id2) -> new_id
    for i in range(num_merges):
        stats = get_stats(ids)
        print("STATS: ")
        print(stats)
        if not stats:
            break                          # nothing left to merge
        pair = max(stats, key=stats.get)   # most frequent pair
        print(pair)
        new_id = 256 + i                   # next available id
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id
        print("MERGES: ")
        print(merges)

        if i % 100 == 0:
            print(f"merge {i}: {pair} -> {new_id} (had {stats[pair]} occurrences)")

    return merges, ids

### **Step 3** — encode and decode using the learned merges

In [ ]:
def bpe_encode(text, merges):
    """
    Apply learned merges to new text, in the order they were learned.
    """
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # Find the pair that was merged EARLIEST in training (lowest new_id)
        # — earlier merges take priority, same order they were learned
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                        # no more applicable merges
        ids = merge(ids, pair, merges[pair])
    return ids

def bpe_decode(ids, merges):
    """
    Reverse the merges to get back raw bytes, then decode to text.
    """
    # Build id -> bytes mapping, starting from the 256 raw bytes
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for (p0, p1), idx in merges.items():
        vocab[idx] = vocab[p0] + vocab[p1]   # concatenate the two pieces

    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode("utf-8", errors="replace")

### **Step 4** — Wiring it into MiniGPT

In [ ]:
import requests

# url = "https://www.gutenberg.org/files/19616/19616-0.txt"  # Aesop's Fables, Vol 1 - small plain text
# response = requests.get(url)
# text = response.text

text = """
A few lines will suffice to explain why we have compiled the present volume, to what wants it responds, and what its sphere of usefulness may possibly embrace.

In our teaching of plastic anatomy, especially at the École des Beaux-Arts—where, for the past nine years, we have had the very great honour of supplementing the teaching of our distinguished master, Mathias Duval, after having been prosector for his course of lectures since 1881—it is our practice to give, as a complement to the study of human anatomy, a certain number of lessons on the anatomy of those animals which artists might be called on to represent.

Now, we were given to understand that the subject treated in our lectures interested our hearers, so much so that we were not surprised to learn that a certain number repeatedly expressed a desire to see these lectures united in book form.

To us this idea was not new; for many years the work in question had been in course of preparation, and we had collected materials for it, with the object of filling up a void of which the existence was to be regretted. But our many engagements prevented us from executing our project as early as we would have wished. It is this work which we publish to-day.

Putting aside for a moment the wish expressed by our hearers, we feel ourselves in duty bound to inquire whether the utility of this publication is self-evident. Let it be clearly understood that we wish to express here our opinion[vii] on this subject, while putting aside every personal sentiment of an author.

No one now disputes the value of anatomical studies made in view of carrying out the artistic representation of man. Nevertheless—for we must provide against all contingencies—the conviction on this subject may be more or less absolute; and yet it must possess this character in an intense degree in order that these studies may be profitable, and permit the attainment of the goal which is proposed in undertaking them. It is in this way that we ever strive to train the students whose studies we direct; not only to admit the value of these studies, but to be materially and deeply convinced of the fact without any restriction. Such is the sentiment which we endeavour to create and vigorously encourage. And we may be permitted to add that we have often been successful in this direction.

Therefore it is that, at the beginning of our lectures, and in anticipation of possible objections, we are accustomed to take up the question of the utility of plastic anatomy. And in so doing, it is in order to combat at the outset the idea—as mischievous as it is false—which is sometimes imprudently enunciated, that the possession of scientific knowledge is likely to tarnish the purity and freshness of the impressions received by the artist, and to place shackles on the emotional sincerity of their representation.
"""

print(f"Total characters: {len(text):,}")
print(f"First 300 chars:\n{text[:300]}")

In [ ]:
# Train once, on any dataset
merges, _ = train_bpe(text, vocab_size=2000)   # pick your target vocab size
vocab_size = 256 + len(merges)

encode = lambda s: bpe_encode(s, merges)
decode = lambda l: bpe_decode(l, merges)

# Test it
sample = "To be or not to be"
tokens = encode(sample)
print(f"Tokens: {tokens}")
print(f"Decoded: {decode(tokens)}")   # should exactly reconstruct the original